In [ ]:
%load_ext autoreload
%autoreload 2
import os
import numpy as np
import matplotlib.pyplot as plt
import richio
import unyt as u

plt.style.use("/data2/yujiehe/rich-monitor/academic-mplstyle/nice.mplstyle")
DATADIR = "/disks/emrdata/"

In [ ]:
import h5py
import re
import glob

SS24DIR = os.path.join(DATADIR, "SS24")

Rstar = 1 * richio.units.lscale
Mstar = 1 * richio.units.mscale
Mbh = 1e6 * richio.units.mscale
r_p = Rstar * (Mbh/Mstar)**(1/3) * 1
tmin = np.pi/np.sqrt(2) * (Rstar**3/u.G/Mstar)**(1/2)*(Mbh/Mstar)**(1/2)
Delta = u.G * Mbh / (4 * r_p) * Mstar

In [ ]:
# Old run, split across 3 directories with old-style per-rank HDF5 snapshots
# (snap_<N>.h5, not snap_full_<N>.h5). TEMPTDE4_new is a redo of TEMPTDE4's tail
# (snapshots 820-952 exist in both) - it wins on overlap since it's the later run.
SS24_RUNDIRS = ["TEMPTDE", "TEMPTDE4", "TEMPTDE4_new"]

snap_paths = {}
for rundir in SS24_RUNDIRS:
    for path in glob.glob(os.path.join(SS24DIR, rundir, "snap_*.h5")):
        snapnum = int(re.search(r"snap_(\d+)\.h5$", path).group(1))
        snap_paths[snapnum] = path

snapnums = sorted(snap_paths)
print(f"{len(snapnums)} snapshots, {snapnums[0]}..{snapnums[-1]}")

In [ ]:
def load_slab(path, fields, slice_coord=0, plane_axis="Z"):
    """Stream an old-format (per-rank) snapshot rank-by-rank, keeping only cells
    within one cell-size of the slice plane. A plain field read (snap.<field>)
    concatenates every rank first - up to ~2e8 cells for the late SS24
    snapshots - which OOMs this box, so filter before concatenating instead.
    """
    in_plane = [ax for ax in "XYZ" if ax != plane_axis]
    keys = in_plane + list(fields)
    chunks = {k: [] for k in keys}
    with h5py.File(path, "r") as f:
        nrank = max(int(k[4:]) for k in f if k.startswith("rank")) + 1
        for i in range(nrank):
            g = f[f"rank{i}"]
            normal = g[plane_axis][()]
            vol = g["Volume"][()]
            mask = np.abs(normal - slice_coord) < vol ** (1 / 3)
            if not mask.any():
                continue
            for k in keys:
                chunks[k].append(g[k][()][mask])
    return {
        k: np.concatenate(v) * richio.units.get_unit(k, u.Dimensionless)
        for k, v in chunks.items()
    }

In [ ]:
# Sanity check: midplane density slab of the latest snapshot
snap = richio.load(snap_paths[snapnums[-1]])
print("t =", snap.t, " t/tmin =", snap.t[0]/tmin)

slab = load_slab(snap_paths[snapnums[-1]], fields=["Density"])
print({k: v.shape for k, v in slab.items()})

fig, ax = plt.subplots()
sc = ax.scatter(slab["X"], slab["Y"], c=np.log10(slab["Density"].in_cgs().v), s=2, cmap="inferno", linewidths=0)
ax.set_aspect("equal")
fig.colorbar(sc, label=r"$\log_{10}\rho\,[\mathrm{g/cm^3}]$")